# ModelPilot User Tutorial
## ML Workflow Optimizer

**ModelPilot** is an AI-powered developer tool designed to help machine learning engineers manage the model lifecycle from experimentation through deployment and monitoring.

The application combines **Python, Flask, SQLite, D3.js, scikit-learn, and the Anthropic API** to provide model management, experiment tracking, AI-generated insights, hyperparameter recommendations, deployment controls, predictions, and monitoring.

### ModelPilot Workflow

**Train → Track → Analyze → Optimize → Deploy → Predict → Monitor**

This notebook explains how to use the final ModelPilot application and how its major components work.

## 1. Environment and Setup

ModelPilot requires Python 3.10 or later. The project uses a virtual environment so dependencies can be isolated from other Python projects.

Major technologies include:

- Python
- Flask
- scikit-learn
- Anthropic API
- SQLite
- D3.js
- HTML/CSS/JavaScript
- Google Cloud Run

API credentials such as the Anthropic API key should be stored as environment variables and should never be committed to GitHub.

In [ ]:
# Example local environment setup
# Run these commands in a terminal rather than exposing API credentials here.

# python3 -m venv venv312
# source venv312/bin/activate
# pip install -r requirements.txt

print("ModelPilot environment setup documentation loaded.")

## 2. Dataset and Machine Learning Task

ModelPilot's demonstration workflow uses a cleaned **hotel booking dataset**.

The machine learning task is **binary classification**: predicting whether a hotel reservation will be canceled.

Data preprocessing includes handling missing values, removing inappropriate or leakage-related fields, creating useful features, and preparing categorical variables for model training.

Two engineered features used by the project are:

- `total_guests`
- `total_nights`

The final training workflow compares multiple classification algorithms.

## 3. Final Baseline Models

Three final baseline versions are used throughout ModelPilot:

| Version | Algorithm | Test Accuracy | Macro-F1 | ROC-AUC | Train-Test Gap |
|---|---|---:|---:|---:|---:|
| v7 | Logistic Regression | 81.18% | 0.7883 | 0.8883 | -0.18% |
| v8 | Decision Tree | 83.22% | 0.8176 | 0.9100 | 1.86% |
| v9 | K-Nearest Neighbors | 80.68% | 0.7801 | 0.8789 | 19.32% |

These versions provide a stable comparison set for deployment analysis and AI-generated recommendations.

Decision Tree v8 achieved the strongest overall baseline test performance, while KNN v9 showed a much larger train-test gap.

## 4. Model Registry and Versioning

ModelPilot uses **SQLite** to maintain a model registry.

The registry records model versions and metadata so engineers can compare models and control which version is used in production.

The application separates model training from deployment. Creating a new model version does **not** mean that the model must immediately replace the current production model.

This supports a safer ML workflow:

1. Train a candidate.
2. Evaluate its metrics.
3. Register the version.
4. Compare it with existing models.
5. Decide whether it should be promoted.

## 5. Automated Experiment Tracking

ModelPilot automatically records experiment information using Python.

Experiment records include information such as:

- Model/algorithm
- Hyperparameters
- Evaluation metrics
- Experiment metadata

Experiment information is stored in **JSON format**, allowing experiment history to be reviewed by the application and analyzed by the AI integration.

The **Experiments** page provides a user-friendly view of these results.

## 6. Real-Time Model Monitoring

The **Monitoring** page displays prediction activity generated by models used in the application.

Python records prediction events, while **D3.js** is used to create interactive visualizations of monitoring data.

The dashboard allows an engineer to observe information such as:

- Prediction activity
- Active model versions
- Model stage
- Prediction latency

A model only develops monitoring activity after it is actually used for predictions. Model versions are not given artificial monitoring data simply because they exist in the registry.

## 7. Anthropic AI Insights

The **AI Insights** page integrates the Anthropic API.

ModelPilot provides model performance information to the LLM and requests structured analysis. The resulting insights help engineers interpret experiment results rather than relying only on raw metrics.

The AI functionality includes:

- Performance observations
- Optimization recommendations
- Hyperparameter suggestions
- Deployment-readiness feedback

The AI is used as a decision-support tool. Engineers remain in control of model training and deployment decisions.

## 8. AI Hyperparameter Optimization

ModelPilot can turn an AI recommendation into a real training experiment.

The workflow is:

1. Anthropic analyzes the final model results.
2. The AI selects an optimization opportunity.
3. It recommends a model and concrete hyperparameters.
4. The engineer reviews the recommendation.
5. The engineer selects **Accept & Retrain**.
6. Flask sends the accepted configuration to the retraining workflow.
7. The selected model is trained using those parameters.
8. The candidate is evaluated.
9. A new model version is registered.

This connects LLM-generated recommendations directly to the machine learning training pipeline.

## 9. Example: AI-Retrained v10

During testing, ModelPilot generated a Decision Tree recommendation using parameters including:

- `max_depth = 10`
- `min_samples_leaf = 4`
- `criterion = "entropy"`

Accepting the AI recommendation triggered the Flask retraining endpoint and produced **v10** as a new local candidate model.

### Comparison

| Metric | Original Decision Tree v8 | AI-Retrained v10 |
|---|---:|---:|
| Test Accuracy | 83.22% | 82.84% |
| Macro-F1 | 0.8176 | 0.8157 |
| ROC-AUC | 0.9100 | 0.9093 |
| Train-Test Gap | 1.86% | 0.41% |

The AI configuration did not outperform v8 on every test metric, but it substantially reduced the train-test gap while maintaining similar predictive performance.

This demonstrates why ModelPilot does not automatically promote every AI-generated candidate to Production.

## 10. Accept & Retrain API Workflow

The AI Insights interface communicates with a Flask endpoint when a user accepts a hyperparameter suggestion.

Conceptually, the request contains the recommended model and parameter configuration:

In [ ]:
# Simplified example of the information sent by the interface.
# This cell does NOT trigger retraining.

example_ai_suggestion = {
    "model": "Decision Tree",
    "parameters": {
        "max_depth": 10,
        "min_samples_leaf": 4,
        "criterion": "entropy"
    }
}

example_ai_suggestion

The application validates the suggestion before training. Only supported models and approved hyperparameters are accepted.

This helps prevent arbitrary LLM output from being passed directly into the training pipeline.

## 11. Deployment Workflow

Training and deployment are intentionally separate.

The **Deployment** functionality allows an engineer to decide which evaluated model should serve as the Production version.

A typical workflow is:

**Candidate → Evaluation → Deployment Decision → Production**

This human-in-the-loop approach prevents an AI recommendation from automatically replacing a known production model without evaluation.

## 12. Prediction Demo

The **Demo** page provides an interface for running hotel cancellation predictions.

Prediction requests are sent to the Flask backend, which loads the appropriate registered model and returns a prediction.

This demonstrates that ModelPilot is not only displaying stored model metrics—the registered models can actually be used for inference.

## 13. Monitoring After Deployment

Once a deployed model receives prediction requests, ModelPilot records activity that can be viewed through the Monitoring dashboard.

This completes the operational workflow:

**Deploy → Predict → Record Activity → Monitor**

Monitoring allows engineers to observe how registered models are being used after deployment.

## 14. Testing

ModelPilot includes automated tests for application functionality and endpoints.

The project uses Python testing to verify that important components behave reliably before deployment.

During final development, the project's test suite completed successfully with **6 passing tests**.

Testing is especially important because ModelPilot connects several components:

- Flask endpoints
- Model registry
- Model serving
- Experiment tracking
- AI-assisted workflows

## 15. Cloud Deployment

ModelPilot is deployed using **Google Cloud Run**.

Cloud deployment provides a web-accessible version of the Flask application so the ML workflow can be demonstrated outside the local development environment.

The deployment process includes:

1. Push tested source code to the main GitHub branch.
2. Build the application and install dependencies.
3. Start the Flask application using the production server configuration.
4. Deploy a Cloud Run revision.
5. Test the public application pages and workflows.

Secrets such as API keys are configured through the deployment environment rather than stored directly in source code.

## 16. ModelPilot Page Guide

### Dashboard
Provides an overview of the ML workflow and application status.

### Models
Displays final model performance and registered model versions.

### Experiments
Shows recorded experiment results and configurations.

### Monitoring
Visualizes prediction activity and model behavior.

### AI Insights
Uses Anthropic to analyze model performance, provide deployment-readiness feedback, and suggest hyperparameters.

### Deployment
Provides model deployment/version controls.

### Demo
Allows the user to interact with deployed model prediction functionality.

## 17. Complete User Workflow

An ML engineer can use ModelPilot as follows:

1. **Train** models using the ML pipeline.
2. **Track** experiments automatically.
3. **Compare** model performance.
4. **Analyze** results using Anthropic AI Insights.
5. **Optimize** a model using an AI-generated hyperparameter suggestion.
6. **Evaluate** the newly registered candidate.
7. **Deploy** an approved model version.
8. **Predict** using the deployed model.
9. **Monitor** prediction activity through the D3.js dashboard.

### Final Workflow

**Train → Track → Analyze → Optimize → Evaluate → Deploy → Predict → Monitor**

## 18. Conclusion

ModelPilot integrates the major stages of an ML engineering workflow into one application.

Rather than treating model training, experiment tracking, AI analysis, deployment, prediction, and monitoring as unrelated tasks, ModelPilot connects them through a Flask-based interface.

The project demonstrates how an LLM such as Anthropic can support ML engineers by analyzing experimental results and proposing new configurations while still keeping model evaluation and production deployment under human control.